# 3. Deskriptiv statistik

Deskriptiv statistik **beskriver** den data vi har framför oss, med tabeller, diagram och några få sammanfattande tal.
Vi drar ännu inga slutsatser om någon större population – det kommer senare.

Vi börjar med sju tal som du kan räkna på i huvudet, och går sedan över till en större datamängd.

**Efter den här notebooken kan du**
- räkna ut och tolka medelvärde, median och typvärde, och välja rätt mått
- räkna ut och tolka variationsbredd, standardavvikelse, kvartiler och kvartilavstånd
- göra frekvenstabeller, stapeldiagram, histogram och lådagram
- förklara varför ett extremvärde påverkar vissa mått men inte andra

**Förkunskaper:** [2. Variabeltyper och skalnivåer](../02_variabeltyper/02_variabeltyper.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## 3.1 Lägesmått: var ligger värdena?

Sju personer i ett fikarum har följande månadslöner (tkr):

In [ ]:
salaries = pd.Series([28, 31, 31, 33, 35, 38, 42])

print("Medelvärde:", salaries.mean())       # summan delat med antalet
print("Median:    ", salaries.median())     # det mittersta värdet när talen är sorterade
print("Typvärde:  ", salaries.mode()[0])    # det vanligaste värdet

Kontrollera gärna för hand: summan är 238 och 238 / 7 = 34. Det mittersta av sju sorterade tal är det fjärde, alltså 33.

### Vad händer när chefen kommer in?

🔮 **Gissa först:** Vd:n, som tjänar 250 tkr, kliver in i fikarummet. Vad händer med medelvärdet? Med medianen?

In [ ]:
visitor = 250        # ← ändra: pröva 50, 250 och 1000

with_visitor = pd.concat([salaries, pd.Series([visitor])], ignore_index=True)

print(f"Utan besökare:  medelvärde {salaries.mean():.1f}   median {salaries.median():.1f}")
print(f"Med besökare:   medelvärde {with_visitor.mean():.1f}   median {with_visitor.median():.1f}")

🧪 **Labba:** Pröva olika värden på `visitor`. Hur högt kan du få medelvärdet? Hur mycket rör sig medianen?

Medelvärdet använder **alla värdens storlek** och dras därför med av ett extremvärde. Medianen bryr sig bara om
**ordningen** och kallas därför **robust**. Med vd:n i rummet tjänar "medelpersonen" mer än sju av åtta faktiskt gör –
då är medianen en ärligare beskrivning av en typisk person.

> **Tumregel:** symmetrisk data utan extremvärden → medelvärde. Sned data eller extremvärden → median. Redovisa gärna båda.

✏️ **Din tur:** Sex lägenheter på en gata har sålts för 2.1, 2.4, 2.4, 2.6, 2.9 och 9.5 miljoner kronor.
Räkna ut medelvärde och median. Vilket mått skulle en mäklare som vill locka säljare använda? Vilket beskriver en typisk lägenhet?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
prices = pd.Series([2.1, 2.4, 2.4, 2.6, 2.9, 9.5])
print(f"Medelvärde: {prices.mean():.2f} mnkr")
print(f"Median:     {prices.median():.2f} mnkr")
# Medelvärdet (3,65) dras upp av en enda dyr lägenhet och är högre än fem av sex priser.
# Medianen (2,5) beskriver den typiska lägenheten.
```

</details>

## 3.2 Spridningsmått: hur utspridda är värdena?

Två grupper kan ha samma medelvärde och ändå vara helt olika:

In [ ]:
group_a = pd.Series([33, 34, 34, 34, 35])
group_b = pd.Series([14, 24, 34, 44, 54])

print(f"Grupp A: medelvärde {group_a.mean():.0f}")
print(f"Grupp B: medelvärde {group_b.mean():.0f}")

Vi behöver ett mått på spridningen. Det enklaste är **variationsbredden**: största minus minsta värdet.
Den är lätt att förstå men beror bara på de två mest extrema värdena.

### Standardavvikelsen, steg för steg

**Standardavvikelsen** svarar på frågan: *hur långt från medelvärdet ligger värdena i genomsnitt?* Så här byggs den upp:

1. Räkna ut varje värdes **avvikelse** från medelvärdet.
2. **Kvadrera** avvikelserna (annars tar plus och minus ut varandra).
3. Summera och dela med **n − 1**. Det ger **variansen**.
4. Ta **roten ur**, så att enheten blir densamma som i datan.

$$s = \sqrt{\frac{\sum (x_i - \bar{x})^2}{n - 1}}$$

In [ ]:
values = group_b          # ← ändra till group_a och kör om

steps = pd.DataFrame({"value": values})
steps["deviation"] = steps["value"] - values.mean()
steps["squared"] = steps["deviation"] ** 2
display(steps)

n = len(values)
variance = steps["squared"].sum() / (n - 1)
print(f"Summan av avvikelserna:  {steps['deviation'].sum():.1f}   (alltid 0 – därför kvadrerar vi)")
print(f"Varians:                 {variance:.1f}")
print(f"Standardavvikelse:       {np.sqrt(variance):.2f}")
print(f"Kontroll med pandas:     {values.std():.2f}")

🧪 **Labba:** Byt till `group_a` och kör om. Stämmer standardavvikelsen med din känsla för hur utspridda grupperna är?

> **Varför n − 1?** När vi räknar på ett stickprov och vill skatta spridningen i hela populationen, ger division med *n*
> ett något för litet värde. Division med *n − 1* rättar till det. pandas `.std()` delar med *n − 1* automatiskt.
> NumPys `np.std(x)` delar med *n*, så där skriver man `np.std(x, ddof=1)`.

### Kvartiler och kvartilavstånd

**Kvartilerna** delar de sorterade värdena i fyra lika stora delar: 25 % av värdena ligger under **Q1**, 50 % under medianen
och 75 % under **Q3**. **Kvartilavståndet** Q3 − Q1 är bredden på de mittersta 50 % av värdena.

🔮 **Gissa först:** vilket spridningsmått påverkas *minst* när vd:n kommer in i fikarummet?

In [ ]:
def describe_spread(x, label):
    q1, q3 = x.quantile([0.25, 0.75])
    print(f"{label:<14} variationsbredd {x.max() - x.min():>6.1f}   standardavvikelse {x.std():>6.1f}   kvartilavstånd {q3 - q1:>5.1f}")

describe_spread(salaries, "Utan vd:")
describe_spread(with_visitor, "Med vd:")

Kvartilavståndet är robust, precis som medianen. Variationsbredd och standardavvikelse påverkas kraftigt av ett enda extremvärde.

## 3.3 En större datamängd

Nu räcker det inte att titta på talen ett och ett. Vi simulerar 200 personer:

In [ ]:
n_people = 200
people = pd.DataFrame({
    "height": rng.normal(172, 9, n_people).round(1),                    # cm
    "num_children": rng.poisson(1.5, n_people),
    "education": rng.choice(["Grundskola", "Gymnasium", "Högskola"], n_people, p=[0.2, 0.45, 0.35]),
    "income": rng.lognormal(np.log(32), 0.4, n_people).round(1),        # tkr per månad
})
people.head()

### Frekvenstabell och stapeldiagram – för kvalitativa och diskreta variabler

En **frekvenstabell** visar hur många gånger varje värde förekommer. **Relativ frekvens** är samma sak som andel.

In [ ]:
order = ["Grundskola", "Gymnasium", "Högskola"]       # ordinalskala: visa i naturlig ordning
frequency = people["education"].value_counts().reindex(order)

pd.DataFrame({"frekvens": frequency, "relativ frekvens": (frequency / frequency.sum()).round(2)})

In [ ]:
frequency.plot(kind="bar", rot=0, title="Utbildningsnivå")
plt.xlabel("Utbildning")
plt.ylabel("Antal")
plt.show()

✏️ **Din tur:** Gör en frekvenstabell och ett stapeldiagram över `num_children`. Vilket är typvärdet?
Ledtråd: `.value_counts().sort_index()` ger värdena i ordningen 0, 1, 2 …

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
children_freq = people["num_children"].value_counts().sort_index()
print(children_freq)
print("Typvärde:", people["num_children"].mode()[0])

children_freq.plot(kind="bar", rot=0)
plt.xlabel("Antal barn")
plt.ylabel("Antal personer")
plt.show()
```

</details>

### Histogram – för kontinuerliga variabler

En kontinuerlig variabel som längd har nästan bara unika värden, så en vanlig frekvenstabell blir meningslös.
I stället delar vi in värdena i **klasser** (intervall) och räknar hur många som hamnar i varje.

In [ ]:
classes = pd.cut(people["height"], bins=range(140, 211, 10), right=False)
classes.value_counts().sort_index()

Klassen `[160, 170)` innehåller värden från och med 160 upp till, men inte med, 170. Ett **histogram** är denna tabell som bild.
Antalet klasser är ett val du gör, och det påverkar vad du ser:

In [ ]:
num_bins = 12        # ← ändra: pröva 3, 12 och 80

plt.hist(people["height"], bins=num_bins, edgecolor="white")
plt.xlabel("Längd (cm)")
plt.ylabel("Antal")
plt.title(f"{num_bins} klasser")
plt.show()

🧪 **Labba:** Pröva 3, 12 och 80 klasser. För få klasser döljer formen, för många ger mest brus.
Det finns inget "rätt" antal – välj det som visar formen tydligast.

Jämför nu längd och inkomst:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, column, unit in zip(axes, ["height", "income"], ["cm", "tkr per månad"]):
    ax.hist(people[column], bins=20, edgecolor="white")
    ax.axvline(people[column].mean(), color="red", label="medelvärde")
    ax.axvline(people[column].median(), color="black", linestyle="--", label="median")
    ax.set_xlabel(f"{column} ({unit})")
    ax.set_ylabel("Antal")
    ax.legend()
plt.tight_layout()
plt.show()

Längden är ungefär **symmetrisk**: medelvärde och median sammanfaller nästan. Inkomsten är **högersned**: en lång svans
åt höger drar upp medelvärdet över medianen. Det är fikarummet med vd:n igen, fast i större skala.

### Allt på en gång: `describe()`

In [ ]:
people[["height", "num_children", "income"]].describe().round(1)

Raderna `25%`, `50%` och `75%` är Q1, medianen och Q3.

## 3.4 Lådagram

Ett **lådagram** (boxplot) ritar upp kvartilerna:

- **Lådan** går från Q1 till Q3 och rymmer de mittersta 50 % av värdena.
- **Strecket i lådan** är medianen.
- **Morrhåren** går ut till det minsta och största värde som ligger högst 1,5 kvartilavstånd från lådan.
- **Enskilda punkter** utanför morrhåren är **uteliggare** (extremvärden).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].boxplot(people["height"])
axes[0].set_title("Längd")
axes[0].set_ylabel("cm")
axes[1].boxplot(people["income"])
axes[1].set_title("Inkomst")
axes[1].set_ylabel("tkr per månad")
for ax in axes:
    ax.set_xticks([])
plt.tight_layout()
plt.show()

I den högersneda inkomstfördelningen ligger medianen närmare lådans nederkant, det övre morrhåret är längre och
uteliggarna finns nästan bara uppåt.

✏️ **Din tur:** Räkna själv ut vilka inkomster som räknas som uteliggare uppåt, alltså de som är större än
Q3 + 1,5 · kvartilavståndet. Hur många är de? Stämmer det med punkterna i lådagrammet?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
q1, q3 = people["income"].quantile([0.25, 0.75])
upper_limit = q3 + 1.5 * (q3 - q1)
outliers = people[people["income"] > upper_limit]["income"]
print(f"Gräns: {upper_limit:.1f} tkr")
print(f"Antal uteliggare uppåt: {len(outliers)}")
print(outliers.sort_values().to_numpy())
```

</details>

## Sammanfattning

| Fråga | Mått | Känsligt för extremvärden? |
|---|---|---|
| Var ligger värdena? | Medelvärde | Ja |
| | Median | Nej (robust) |
| | Typvärde | Nej |
| Hur utspridda är de? | Variationsbredd | Mycket |
| | Standardavvikelse | Ja |
| | Kvartilavstånd | Nej (robust) |

| Variabel | Tabell | Diagram |
|---|---|---|
| Kvalitativ eller diskret | Frekvenstabell | Stapeldiagram |
| Kontinuerlig | Klassindelad frekvenstabell | Histogram, lådagram |

### Kontrollfrågor
1. En kommun redovisar att medelinkomsten steg med 5 % medan medianinkomsten var oförändrad. Vad kan ha hänt?
2. Varför passar ett stapeldiagram bättre än ett histogram för antal barn?
3. Två klasser har samma medelpoäng på ett prov, men standardavvikelsen är 4 i den ena och 15 i den andra. Vad säger det?

<details>
<summary>💡 <b>Svar</b></summary>

1. Inkomsterna har ökat bland dem som redan tjänade mest. Det drar upp medelvärdet men flyttar inte den mittersta personen.
2. Antal barn är en diskret variabel med få möjliga värden. Varje värde kan få en egen stapel, och ingen klassindelning behövs.
3. I den första klassen ligger nästan alla nära medelpoängen. I den andra är resultaten mycket ojämna, med både höga och låga poäng.

</details>

**Nästa notebook:** [4. Fördelningar och normalfördelningen](../04_fordelningar/04_fordelningar.ipynb)